# PRÁCTICA 1

* **Autores:** Claudia Cuevas Ruano, Pablo Tejero Lascorz
* **Pareja:** 03

In [1]:
import time
import numpy as np
from ClasificadorKNN import KNN, DistanceMetricKNN
from ClasificadorNaiveBayes import ClasificadorNaiveBayes
from Datos import Datos
from Estandarizador import Estandarizador
from EstrategiaParticionado import ValidacionSimple, ValidacionCruzada, EstrategiaParticionado


DATASET_TELEFONIA = "../datasets/fuga_telefonia.csv"
DATASET_WDBC      = "../datasets/wdbc.csv"

SEED = 42

In [2]:
validators = [
    ("Validacion Cruzada", ValidacionCruzada(5)),
    ("Validacion Simple",  ValidacionSimple(5, 0.3))
]


def validarNB(nb: ClasificadorNaiveBayes, dataset: Datos, validador: EstrategiaParticionado):
    errores = nb.limpiar_numpy(nb.validacion(validador, dataset))
    error_mean = np.mean(errores)
    error_std = np.std(errores)
    print(f"Error: {error_mean:.3f} +- {error_std:.3f}")

# Clasificador Naive-Bayes

Mediciones para `fuga_telefonia.csv`:

In [3]:
np.random.seed(42)

naiveBayes = ClasificadorNaiveBayes()

validacionCruzada = ValidacionCruzada(5)
validacionSimple = ValidacionSimple(5, 0.3)

dataset=Datos(DATASET_TELEFONIA)


# Validacion Cruzada
print("VALIDACION CRUZADA\n------------------")
validarNB(naiveBayes, dataset, validacionCruzada)

# Validacion Simple
print("\nVALIDACION SIMPLE\n-----------------")
validarNB(naiveBayes, dataset, validacionCruzada)

VALIDACION CRUZADA
------------------
Total Time: 0.02413153648376465
Total Time: 0.024827241897583008
Total Time: 0.025159120559692383
Total Time: 0.025058984756469727
Total Time: 0.02541327476501465
Error: 0.335 +- 0.015

VALIDACION SIMPLE
-----------------
Total Time: 0.027478933334350586
Total Time: 0.02904510498046875
Total Time: 0.035732269287109375
Total Time: 0.028731107711791992
Total Time: 0.02315354347229004
Total Time: 0.02546525001525879
Total Time: 0.0235443115234375
Total Time: 0.024768590927124023
Total Time: 0.027324199676513672
Total Time: 0.024910688400268555
Error: 0.331 +- 0.021


Mediciones para `wdbc.csv`:

In [4]:
np.random.seed(42)

naiveBayes = ClasificadorNaiveBayes()

validacionCruzada = ValidacionCruzada(5)
validacionSimple = ValidacionSimple(5, 0.3)

dataset=Datos(DATASET_WDBC)


# Validacion Cruzada
print("VALIDACION CRUZADA\n------------------")
validarNB(naiveBayes, dataset, validacionCruzada)

# Validacion Simple
print("\nVALIDACION SIMPLE\n-----------------")
validarNB(naiveBayes, dataset, validacionCruzada)

VALIDACION CRUZADA
------------------
Total Time: 0.02268671989440918
Total Time: 0.019088029861450195
Total Time: 0.018145084381103516
Total Time: 0.018963098526000977
Total Time: 0.017560958862304688
Error: 0.063 +- 0.018

VALIDACION SIMPLE
-----------------
Total Time: 0.01778435707092285
Total Time: 0.01843094825744629
Total Time: 0.020411014556884766
Total Time: 0.020082712173461914
Total Time: 0.020644187927246094
Total Time: 0.02199411392211914
Total Time: 0.022717952728271484
Total Time: 0.019131183624267578
Total Time: 0.020726680755615234
Total Time: 0.017342090606689453
Error: 0.064 +- 0.013


En la siguiente tabla se muestran los resultados de la clasificación del dataset del archivo `fuga_telefonia.csv` con la implementación propia del clasificador Naive-Bayes:

| **Clasificador** |**Método de medición** | **Laplace** | **`mean(error)`** |
|------------------|-----------------------|-------------|-------------------|
| ClasificadorNaiveBayes | ValidacionCruzada (5 particiones) | Sí | 0.3162 |
| ClasificadorNaiveBayes | ValidacionCruzada (5 particiones) | No | 0.3297 |
| ClasificadorNaiveBayes | ValidacionSimple | Sí | 0.3544 |
| ClasificadorNaiveBayes | ValidacionSimple | No | 0.3273 |
| MultinomialNB | ValidacionCruzada (5 particiones) | Sí | 0.427 |
| MultinomialNB | ValidacionSimple | Sí | 0.427 |
| GaussianNB | ValidacionCruzada (5 particiones) | Sí | 0.441 |
| GaussianNB | ValidacionSimple | Sí | 0.378 |


En la siguiente tabla se muestran los resultados equivalentes para la clasificación del dataset del archivo `wdbc.csv`:

| **Clasificador** | **Método de medición** | **Laplace** | **`mean(error)`** |
|------------------|-----------------------|-------------|-------------------|
| ClasificadorNaiveBayes | ValidacionCruzada (5 particiones) | Sí | 0.0668 |
| ClasificadorNaiveBayes | ValidacionCruzada (5 particiones) | No | 0.0720 |
| ClasificadorNaiveBayes | ValidacionSimple | Sí | 0.3544 |
| ClasificadorNaiveBayes | ValidacionSimple | No | 0.3273 |
| MultinomialNB | ValidacionCruzada (5 particiones) | Sí | 0.1650 |
| MultinomialNB | ValidacionSimple | Sí | 0.061 |
| GaussianNB | ValidacionCruzada (5 particiones) | Sí | 0.1050
| GaussianNB | ValidacionSimple | Sí | 0.064 |


Contemplamos dos implementaciones distintas, ya que el NaiveBayes propio combina MultinomialNB y GaussianNB, usando uno u otro dependiendo del tipo de dato. Observamos que el NaiveBayes propio maneja datos mixtos de manera más efectiva que MultinomialNb de sklearn y que las diferencias entre validación simple y cruzada son moderadas, indicando estabilidad.

La corrección de Laplace evita probabilidades cero en atributos nominales. Quitándolo, observamos un pequeño aumnto del error, el cual aumenta cuanto más mixto sea el dataset.

# Clasificador KNN

## Preprocesamiento de Datos

Para el cálculo de la distancia entre la muestra que se quiere clasificar y una de las muestras del conjunto de entrenamiento, el clasificador KNN debe calcular la diferencia entre los valores del atributo para cada atributo. Para evitar problemas de escalas y posibles sesgos, es necesario aplicar una serie de transformaciones sobre los datos que dependerán de la naturaleza de los mismos.

Para los atributos numéricos y atributos nominales cuantitativos, será necesario aplicar un estandarizado de los datos. Dicho estandarizado se suele realizar a partir de la media y la desviación estándar de los datos.

Para los atributos nominales, en los que no hay una jerarquía entre los posibles valores de la clase, la estandarización no es un procedimiento correcto. Supongamos un caso en el que tenemos un atributo "Animal" que puede tomar los valores [ "Perro", "Gato", "Pez", "Caballo" ] y que codificaremos como [ 0, 1, 2, 3 ], respectivamente. Si aplicamos una estandarización de los datos, el algoritmo de cálculo de distancias de KNN encontrará que la distancia entre "Perro" y "Caballo" será mayor que la que pueda haber entre "Perro" y "Gato". Para solventar este problema, se plantea una solución en la que se divide la columna del atributo en varias columnas, tantas como posibles valores pueda tomar el mismo. Es decir, sustituimos el atributo "Animal" por cuatro nuevos atributos "Animal_Perro", "Animal_Gato", "Animal_Pez" y "Animal_Caballo" y, seguidamente, para cada muestra del conjunto de datos codificamos el valor de cada nuevo atributo como 1 si la muestra es del tipo de animal y 0 si no lo es. Ahora, la distancia entre dos muestras será 0 si los dos animales son del mismo tipo y 1 si los dos animales son de distinto tipo. Este procedimiento es conocido como codificación One Hot.

Se ha implementado la clase `Estandarizador` en `Estandarizador.py` que lleva a cabo este proceso. Entre otras cosas, el constructor recibe un objeto de tipo `Datos` con los datos de referencia para la transformación (para ajustar el estandarizador con la media y la desviación estándar de cada atributo) y una lista con los nombres de las columnas que contienen atributos nominales cuantitativos (por ejemplo, tallas de una camiseta), para que no se aplique la codificación One Hot, ya que sí existe una jerarquía entre valores. También se implementa un método `estandarizarDatos` que recibe un conjunto de datos y aplica las transformaciones. Ejemplo de uso:


``` Python
estandarizador = Estandarizador(datosEntrenamiento, media=True, std=True, exclude=None)
dataTrain = estandarizador.estandarizarDatos(datosEntrenamiento)
dataTest  = estandarizador.estandarizarDatos(datosTest)
```

## Mediciones

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score


def validarKNN(dataset, k, metric):
    knn = KNN(k, metric)
    validador = ValidacionCruzada(5)

    errores = knn.validacion(validador, dataset, SEED)
    error_mean = np.mean(errores)
    error_std = np.std(errores)

    print(f"Precision de las predicciones: {1 - error_mean:.3f}")
    print(f"Error: {error_mean:.3f} +- {error_std:.3f}")


def validarScikitKNN(dataset, k, metric):
    clases = dataset.datos.iloc[:,-1].values

    knn = KNeighborsClassifier(n_neighbors=k, metric=metric, weights="uniform")

    t_start = time.time()
    scores = cross_val_score(knn, dataset.datos, clases, cv=5)
    t_end = time.time()

    print("Total Time:", t_end - t_start)
    print(f"Error: {np.mean(scores):.3f} +- {np.std(scores):.3f}")

### Con estandarizado de datos

Mediciones para `fuga_telefonia.csv` con la implementación propia de KNN y la versión de scikit utilizando la métrica "euclides":

In [6]:
dataset = Datos(DATASET_TELEFONIA)

# Preprocesamiento de datos
estandarizador = Estandarizador(dataset, True, True)
ppdataset = estandarizador.estandarizarDatos(dataset.datos)


print("KNN (K=5, metric='euclidean')\n------------------")
validarKNN(dataset, 5, DistanceMetricKNN.EUCLIDES)

print("\nKNN (K=11, metric='euclidean')\n------------------")
validarKNN(dataset, 11, DistanceMetricKNN.EUCLIDES)

print("\nScikit KNN (K=5, metric='euclidean')\n------------------")
validarScikitKNN(dataset, 5, "euclidean")

print("\nScikit KNN (K=11, metric='euclidean')\n------------------")
validarScikitKNN(dataset, 11, "euclidean")

KNN (K=5, metric='euclidean')
------------------
Tiempo Calculo Distancias (sec): 24.129730224609375
Tiempo Calculo vecinos (sec): 0.07677459716796875
Total Time: 24.22786855697632
Tiempo Calculo Distancias (sec): 20.445170164108276
Tiempo Calculo vecinos (sec): 0.06730198860168457
Total Time: 20.52680468559265
Tiempo Calculo Distancias (sec): 20.352930545806885
Tiempo Calculo vecinos (sec): 0.06783342361450195
Total Time: 20.433284044265747
Tiempo Calculo Distancias (sec): 20.42422890663147
Tiempo Calculo vecinos (sec): 0.06831645965576172
Total Time: 20.50727605819702
Tiempo Calculo Distancias (sec): 20.385412454605103
Tiempo Calculo vecinos (sec): 0.0652778148651123
Total Time: 20.463667392730713
Precision de las predicciones: 0.982
Error: 0.018 +- 0.004

KNN (K=11, metric='euclidean')
------------------
Tiempo Calculo Distancias (sec): 20.277390480041504
Tiempo Calculo vecinos (sec): 0.06740975379943848
Total Time: 20.3590190410614
Tiempo Calculo Distancias (sec): 20.26273441314697

Mediciones para `fuga_telefonia.csv` con la implementación propia de KNN y la versión de scikit utilizando la métrica "manhattan":

In [7]:
dataset = Datos(DATASET_TELEFONIA)

# Preprocesamiento de datos
estandarizador = Estandarizador(dataset, True, True)
ppdataset = estandarizador.estandarizarDatos(dataset.datos)


print("KNN (K=5, metric='manhattan')\n------------------")
validarKNN(dataset, 5, DistanceMetricKNN.MANHATTAN)

print("\nKNN (K=11, metric='manhattan')\n------------------")
validarKNN(dataset, 11, DistanceMetricKNN.MANHATTAN)

print("\nScikit KNN (K=5, metric='manhattan')\n------------------")
validarScikitKNN(dataset, 5, "manhattan")

print("\nScikit KNN (K=11, metric='manhattan')\n------------------")
validarScikitKNN(dataset, 11, "manhattan")

KNN (K=5, metric='manhattan')
------------------
Tiempo Calculo Distancias (sec): 17.768211841583252
Tiempo Calculo vecinos (sec): 0.05172538757324219
Total Time: 17.83581233024597
Tiempo Calculo Distancias (sec): 16.177778482437134
Tiempo Calculo vecinos (sec): 0.04555249214172363
Total Time: 16.236533880233765
Tiempo Calculo Distancias (sec): 16.125865697860718
Tiempo Calculo vecinos (sec): 0.04702019691467285
Total Time: 16.185457468032837
Tiempo Calculo Distancias (sec): 19.804227828979492
Tiempo Calculo vecinos (sec): 0.06497049331665039
Total Time: 19.883654832839966
Tiempo Calculo Distancias (sec): 18.49209761619568
Tiempo Calculo vecinos (sec): 0.06591248512268066
Total Time: 18.573042631149292
Precision de las predicciones: 0.987
Error: 0.013 +- 0.006

KNN (K=11, metric='manhattan')
------------------
Tiempo Calculo Distancias (sec): 17.948941230773926
Tiempo Calculo vecinos (sec): 0.06652259826660156
Total Time: 18.03138566017151
Tiempo Calculo Distancias (sec): 17.9298102855

Mediciones para `wdbc.csv` con la implementación propia de KNN y la versión de scikit utilizando la métrica "euclides":

> **NOTA:** Por algún motivo que no hemos logrado averiguar, cuando se hace la validación con `KNeighborsClassifier` de sklearn salta una excepción. El mensaje de error no ayuda a encontrar el problema.

In [4]:
dataset = Datos(DATASET_WDBC)

# Preprocesamiento de datos
estandarizador = Estandarizador(dataset, True, True)
ppdataset = estandarizador.estandarizarDatos(dataset.datos)


print("KNN (K=5, metric='euclidean')\n------------------")
validarKNN(dataset, 5, DistanceMetricKNN.EUCLIDES)

print("\nKNN (K=11, metric='euclidean')\n------------------")
validarKNN(dataset, 11, DistanceMetricKNN.EUCLIDES)

print("\nScikit KNN (K=5, metric='euclidean')\n------------------")
validarScikitKNN(dataset, 5, "euclidean")

print("\nScikit KNN (K=11, metric='euclidean')\n------------------")
validarScikitKNN(dataset, 11, "euclidean")

KNN (K=5, metric='euclidean')
------------------
Tiempo Calculo Distancias (sec): 4.40053129196167
Tiempo Calculo vecinos (sec): 0.024947643280029297
Total Time: 4.4332356452941895
Tiempo Calculo Distancias (sec): 4.543426275253296
Tiempo Calculo vecinos (sec): 0.025336503982543945
Total Time: 4.5776731967926025
Tiempo Calculo Distancias (sec): 4.486921787261963
Tiempo Calculo vecinos (sec): 0.026485204696655273
Total Time: 4.522427558898926
Tiempo Calculo Distancias (sec): 4.666082382202148
Tiempo Calculo vecinos (sec): 0.025812625885009766
Total Time: 4.701188087463379
Tiempo Calculo Distancias (sec): 4.429428339004517
Tiempo Calculo vecinos (sec): 0.026427268981933594
Total Time: 4.465934753417969
Precision de las predicciones: 0.995
Error: 0.005 +- 0.007

KNN (K=11, metric='euclidean')
------------------
Tiempo Calculo Distancias (sec): 4.162053346633911
Tiempo Calculo vecinos (sec): 0.023639678955078125
Total Time: 4.194017171859741
Tiempo Calculo Distancias (sec): 4.2837326526641

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/home/pablo/Universidad/Curso4/faa/Practicas/p1/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/pablo/Universidad/Curso4/faa/Practicas/p1/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/pablo/Universidad/Curso4/faa/Practicas/p1/.venv/lib/python3.12/site-packages/sklearn/neighbors/_classification.py", line 239, in fit
    return self._fit(X, y)
           ^^^^^^^^^^^^^^^
  File 

Mediciones para `wdbc.csv` con la implementación propia de KNN y la versión de scikit utilizando la métrica "manhattan":

In [ ]:
dataset = Datos(DATASET_WDBC)

# Preprocesamiento de datos
estandarizador = Estandarizador(dataset, True, True)
ppdataset = estandarizador.estandarizarDatos(dataset.datos)


print("KNN (K=5, metric='manhattan')\n------------------")
validarKNN(dataset, 5, DistanceMetricKNN.MANHATTAN)

print("\nKNN (K=11, metric='manhattan')\n------------------")
validarKNN(dataset, 11, DistanceMetricKNN.MANHATTAN)

KNN (K=5, metric='manhattan')
------------------
Tiempo Calculo Distancias (sec): 3.950636148452759
Tiempo Calculo vecinos (sec): 0.02160191535949707
Total Time: 3.9817423820495605
Tiempo Calculo Distancias (sec): 4.335301160812378
Tiempo Calculo vecinos (sec): 0.028231143951416016
Total Time: 4.3731689453125
Tiempo Calculo Distancias (sec): 4.10448145866394
Tiempo Calculo vecinos (sec): 0.02541375160217285
Total Time: 4.139776706695557
Tiempo Calculo Distancias (sec): 4.029597043991089
Tiempo Calculo vecinos (sec): 0.02321648597717285
Total Time: 4.061283349990845
Tiempo Calculo Distancias (sec): 4.289044141769409
Tiempo Calculo vecinos (sec): 0.02602362632751465
Total Time: 4.324463129043579
Precision de las predicciones: 0.995
Error: 0.005 +- 0.007

KNN (K=11, metric='manhattan')
------------------
Tiempo Calculo Distancias (sec): 3.961287021636963
Tiempo Calculo vecinos (sec): 0.02147531509399414
Total Time: 3.9929401874542236
Tiempo Calculo Distancias (sec): 3.8269219398498535
Tie

### Sin estandarizado de datos

Mediciones para `fuga_telefonia.csv` con la implementación propia de KNN y la versión de scikit utilizando la métrica "euclides":

In [8]:
dataset = Datos(DATASET_TELEFONIA)


print("KNN (K=5, metric='euclidean')\n------------------")
validarKNN(dataset, 5, DistanceMetricKNN.EUCLIDES)

print("\nKNN (K=11, metric='euclidean')\n------------------")
validarKNN(dataset, 11, DistanceMetricKNN.EUCLIDES)

print("\nScikit KNN (K=5, metric='euclidean')\n------------------")
validarScikitKNN(dataset, 5, "euclidean")

print("\nScikit KNN (K=11, metric='euclidean')\n------------------")
validarScikitKNN(dataset, 11, "euclidean")

KNN (K=5, metric='euclidean')
------------------
Tiempo Calculo Distancias (sec): 16.38129496574402
Tiempo Calculo vecinos (sec): 0.05530118942260742
Total Time: 16.447600603103638
Tiempo Calculo Distancias (sec): 15.387327194213867
Tiempo Calculo vecinos (sec): 0.05207657814025879
Total Time: 15.4513418674469
Tiempo Calculo Distancias (sec): 15.656980037689209
Tiempo Calculo vecinos (sec): 0.05444645881652832
Total Time: 15.722206354141235
Tiempo Calculo Distancias (sec): 15.371126651763916
Tiempo Calculo vecinos (sec): 0.05097627639770508
Total Time: 15.432687759399414
Tiempo Calculo Distancias (sec): 15.534797430038452
Tiempo Calculo vecinos (sec): 0.05247759819030762
Total Time: 15.598195314407349
Precision de las predicciones: 0.982
Error: 0.018 +- 0.004

KNN (K=11, metric='euclidean')
------------------
Tiempo Calculo Distancias (sec): 15.4002206325531
Tiempo Calculo vecinos (sec): 0.0530247688293457
Total Time: 15.463858127593994
Tiempo Calculo Distancias (sec): 16.1807560920715

Mediciones para `fuga_telefonia.csv` con la implementación propia de KNN y la versión de scikit utilizando la métrica "manhattan":

In [9]:
dataset = Datos(DATASET_TELEFONIA)


print("KNN (K=5, metric='manhattan')\n------------------")
validarKNN(dataset, 5, DistanceMetricKNN.MANHATTAN)

print("\nKNN (K=11, metric='manhattan')\n------------------")
validarKNN(dataset, 11, DistanceMetricKNN.MANHATTAN)

print("\nScikit KNN (K=5, metric='manhattan')\n------------------")
validarScikitKNN(dataset, 5, "manhattan")

print("\nScikit KNN (K=11, metric='manhattan')\n------------------")
validarScikitKNN(dataset, 11, "manhattan")

KNN (K=5, metric='manhattan')
------------------
Tiempo Calculo Distancias (sec): 14.408986806869507
Tiempo Calculo vecinos (sec): 0.05093050003051758
Total Time: 14.47081732749939
Tiempo Calculo Distancias (sec): 13.53598403930664
Tiempo Calculo vecinos (sec): 0.0482630729675293
Total Time: 13.59577488899231
Tiempo Calculo Distancias (sec): 13.5147705078125
Tiempo Calculo vecinos (sec): 0.049423933029174805
Total Time: 13.575473308563232
Tiempo Calculo Distancias (sec): 13.538970232009888
Tiempo Calculo vecinos (sec): 0.04950404167175293
Total Time: 13.599068403244019
Tiempo Calculo Distancias (sec): 13.503502368927002
Tiempo Calculo vecinos (sec): 0.04475808143615723
Total Time: 13.55825400352478
Precision de las predicciones: 0.987
Error: 0.013 +- 0.006

KNN (K=11, metric='manhattan')
------------------
Tiempo Calculo Distancias (sec): 13.398548364639282
Tiempo Calculo vecinos (sec): 0.047689199447631836
Total Time: 13.45631456375122
Tiempo Calculo Distancias (sec): 13.4588010311126

Mediciones para `wdbc.csv` con la implementación propia de KNN y la versión de scikit utilizando la métrica "euclides":

In [11]:
dataset = Datos(DATASET_WDBC)


print("KNN (K=5, metric='euclidean')\n------------------")
validarKNN(dataset, 5, DistanceMetricKNN.EUCLIDES)

print("\nKNN (K=11, metric='euclidean')\n------------------")
validarKNN(dataset, 11, DistanceMetricKNN.EUCLIDES)

KNN (K=5, metric='euclidean')
------------------
Tiempo Calculo Distancias (sec): 4.347533226013184
Tiempo Calculo vecinos (sec): 0.023771047592163086
Total Time: 4.381112337112427
Tiempo Calculo Distancias (sec): 4.318737030029297
Tiempo Calculo vecinos (sec): 0.022505998611450195
Total Time: 4.350534915924072
Tiempo Calculo Distancias (sec): 4.163990497589111
Tiempo Calculo vecinos (sec): 0.02216029167175293
Total Time: 4.195544481277466
Tiempo Calculo Distancias (sec): 4.200051546096802
Tiempo Calculo vecinos (sec): 0.02149677276611328
Total Time: 4.22966194152832
Tiempo Calculo Distancias (sec): 4.4691972732543945
Tiempo Calculo vecinos (sec): 0.02203536033630371
Total Time: 4.500286102294922
Precision de las predicciones: 0.995
Error: 0.005 +- 0.007

KNN (K=11, metric='euclidean')
------------------
Tiempo Calculo Distancias (sec): 4.325755596160889
Tiempo Calculo vecinos (sec): 0.02055215835571289
Total Time: 4.355349540710449
Tiempo Calculo Distancias (sec): 4.257227420806885
Ti

Mediciones para `wdbc.csv` con la implementación propia de KNN y la versión de scikit utilizando la métrica "manhattan":

In [12]:
dataset = Datos(DATASET_WDBC)


print("KNN (K=5, metric='manhattan')\n------------------")
validarKNN(dataset, 5, DistanceMetricKNN.MANHATTAN)

print("\nKNN (K=11, metric='manhattan')\n------------------")
validarKNN(dataset, 11, DistanceMetricKNN.MANHATTAN)

KNN (K=5, metric='manhattan')
------------------
Tiempo Calculo Distancias (sec): 3.9873104095458984
Tiempo Calculo vecinos (sec): 0.02272772789001465
Total Time: 4.019447088241577
Tiempo Calculo Distancias (sec): 3.966076374053955
Tiempo Calculo vecinos (sec): 0.021062612533569336
Total Time: 3.9960670471191406
Tiempo Calculo Distancias (sec): 3.8684637546539307
Tiempo Calculo vecinos (sec): 0.020412921905517578
Total Time: 3.8971714973449707
Tiempo Calculo Distancias (sec): 3.9189343452453613
Tiempo Calculo vecinos (sec): 0.0211029052734375
Total Time: 3.949313163757324
Tiempo Calculo Distancias (sec): 3.822118043899536
Tiempo Calculo vecinos (sec): 0.018414020538330078
Total Time: 3.8490822315216064
Precision de las predicciones: 0.995
Error: 0.005 +- 0.007

KNN (K=11, metric='manhattan')
------------------
Tiempo Calculo Distancias (sec): 3.5918045043945312
Tiempo Calculo vecinos (sec): 0.018280029296875
Total Time: 3.618971586227417
Tiempo Calculo Distancias (sec): 3.5725045204162

### Análisis de resultados

En la siguiente tabla se muestran los resultados de la clasificación del dataset del archivo `fuga_telefonia.csv` con la implementación propia del clasificador Naive-Bayes:


| **Clasificador** | ***K*** | **Estandarizado** | **`mean(error)`** | **`std(error)`** |
|------------------|-----------------------|-------------|-------------------|------------------|
| KNN | 5 | Sí | 0.018 | 0.004 |
| KNN | 5 | No | 0.018 | 0.004 |
| KNN | 11 | Sí | 0.636 | 0.023 |
| KNN | 11 | No | 0.636 | 0.023 |
| KNeighborsClassifier | 5 | Sí | 0.637 | 0.022 |
| KNeighborsClassifier | 11 | Sí | 0.636 | 0.023 |


Es evidente que hemos hecho algo mal. En ningún momento hemos hecho bien la parte de la medición de datos, pero ya no nos da tiempo a corregirlo. Dos posibles errores:
1. Nuestro KNN siempre estandariza los datos.
2. Incluso en las secciones del notebook donde estandarizamos los datos le pasábamos el dataset sin estandarizar al KNN de scikit.
Estos motivos explican la enorme diferencia de error entre los errores de nuestro KNN y los de scikit, siendo la precisión de nuestro clasificador mucho mayor.

Esta entrega se nos ha dado fatal. La próxima vez lo haremos mejor.


#### Notas adicionales

Podemos optimizar KNN cuando la métrica empleada para el calculo de la distancia entre atributos es la distancia de Euclides, ya que no hace falta computar la raíz cuadrada de la suma de las diferencias entre atributos al cuadrado. Es decir, en lugar de calcular la distancia como:
$$ d_E(P, Q) = \sqrt{\sum_{i=1}{n}{(p_1 - q_1)^2 + (p_2 - q_2)^2 + ... + (p_n - q_n)^2}}$$

Sino como:
$$ d_E(P, Q) = {\sum_{i=1}{n}{(p_1 - q_1)^2 + (p_2 - q_2)^2 + ... + (p_n - q_n)^2}}$$


Otra importante optimización podría venir del aumento del grado de paralelismo. Por ejemplo, se podría disponer de un pool de hilos y que cada uno de ellos realice la predicción de las clases sobre una fracción de las muestras, repartidas de forma equitativa entre los distintos workers.